# 02. Feature Pipeline 및 60봉 데이터셋 골격

V1 baseline에 사용할 **OHLC 및 OHLC만으로 재계산 가능한 가격 특징**을 생성하고, 1분 초과 gap을 넘지 않는 연속 60봉 sequence index를 만든다.

이 노트북의 범위:

- 가격·캔들·추세·변동성·모멘텀·시간 특징 생성
- 거래량·거래대금·VWAP·호가·체결 특징 미사용 검증
- 기술지표 warm-up과 1분봉 gap을 고려한 60봉 sequence index 생성
- 미래정보 누수 테스트
- 날짜 기반 train/validation/test split
- train split에만 fit한 `RobustScaler` 검증

> 아직 라벨과 실제 모델 학습은 수행하지 않는다. 전체 `[sample, 60, feature]` 배열도 메모리에 한 번에 만들지 않고 metadata index와 검증용 소규모 batch만 생성한다.

In [1]:
from __future__ import annotations

from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib import font_manager
import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from sklearn.preprocessing import RobustScaler

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 120)
plt.style.use("seaborn-v0_8-whitegrid") if "seaborn-v0_8-whitegrid" in plt.style.available else None
available_fonts = {font.name for font in font_manager.fontManager.ttflist}
for font_name in ["Noto Sans CJK KR", "Noto Sans CJK JP"]:
    if font_name in available_fonts:
        plt.rcParams["font.family"] = font_name
        break
plt.rcParams["axes.unicode_minus"] = False


## 1. 경로와 V1 정책

In [2]:
def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / "AGENT.md").exists() and (candidate / "README.md").exists():
            return candidate
    raise FileNotFoundError("프로젝트 루트를 찾지 못했습니다.")

PROJECT_ROOT = find_project_root(Path.cwd())
DATA_ROOT = (PROJECT_ROOT / "../../data/stock_data").resolve()
RAW_ROOT = DATA_ROOT / "raw"
PROCESSED_ROOT = DATA_ROOT / "processed"
ENRICHED_PATHS = sorted(RAW_ROOT.glob("session_*/*_enriched.csv"))

SEQUENCE_LENGTH = 60
MIN_RUN_LENGTH_FOR_SEQUENCE = 120  # rolling_std_return_60 warm-up 61봉 + 입력 60봉 - 1
PRIMARY_HORIZON_MINUTES = 10
EMBARGO_MINUTES = 10

SPLIT_BY_SESSION = {
    "session_2026-07-07": "train",
    "session_2026-07-08": "train",
    "session_2026-07-09": "train",
    "session_2026-07-10": "train",
    "session_2026-07-13": "validation",
    "session_2026-07-14": "validation",
    "session_2026-07-15": "test",
}

assert ENRICHED_PATHS, f"enriched 파일이 없습니다: {RAW_ROOT}"
assert {path.parent.name for path in ENRICHED_PATHS} <= set(SPLIT_BY_SESSION), "split에 없는 세션이 있습니다."

print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"DATA_ROOT: {DATA_ROOT}")
print(f"enriched files: {len(ENRICHED_PATHS):,}")


PROJECT_ROOT: /home/user/urbandatalab/YSLee/code/Detecting-entry-signals-for-short-term-trades-right-before-a-rapid-price-surge
DATA_ROOT: /home/user/urbandatalab/YSLee/data/stock_data
enriched files: 170


### V1 고정 원칙

```text
입력          = OHLC 파생 가격 특징 + 시간 특징
sequence      = 동일 종목의 연속된 60개 확정 1분봉
gap           = 1분 초과 gap을 넘는 sequence 생성 금지
금지          = volume, notional, VWAP, quote, bid/ask, trade flow
split         = 날짜 기반 train / validation / test
scaler fit    = train split의 고유 bar에만 수행
```

정규장 halt와 무거래 gap은 V1에서 구별하지 않고 둘 다 sequence 경계로 취급한다. 누락 분을 forward fill하지 않는다.

## 2. 최종 입력 feature schema

In [3]:
SMA_WINDOWS = [3, 5, 10, 20, 30, 60]
EMA_WINDOWS = [3, 5, 9, 12, 20, 26]
VOL_WINDOWS = [3, 5, 10, 20, 60]
ATR_WINDOWS = [5, 10, 14, 20]
POSITION_WINDOWS = [5, 10, 20, 60]
RETURN_LAGS = [1, 2, 3, 5, 10, 20]

MODEL_FEATURES = [
    "log_close_price",
    "open_rel_prev_close", "high_rel_prev_close", "low_rel_prev_close", "close_rel_prev_close",
    *[f"log_return_{lag}" for lag in RETURN_LAGS],
    "body_return", "range_return", "upper_wick", "lower_wick", "close_location",
    "gap_from_prev_close", "body_to_range_ratio", "upper_wick_to_range",
    "lower_wick_to_range", "is_bullish", "is_bearish",
    *[f"close_to_sma{window}" for window in SMA_WINDOWS],
    *[f"close_to_ema{window}" for window in EMA_WINDOWS],
    "sma5_to_sma20", "sma10_to_sma30", "ema9_to_ema20", "ema12_to_ema26",
    "sma5_slope_3", "sma20_slope_5", "ema9_slope_3", "ema20_slope_5",
    *[f"rolling_std_return_{window}" for window in VOL_WINDOWS],
    *[f"atr_pct_{window}" for window in ATR_WINDOWS],
    "bb_width_20", "bb_position_20",
    "macd_scaled", "macd_signal_scaled", "macd_hist_scaled",
    "roc_3", "roc_5", "roc_10", "roc_20",
    "rsi_6", "rsi_14", "stochastic_k_14", "stochastic_d_3",
    *[f"distance_from_high_{window}" for window in POSITION_WINDOWS],
    *[f"distance_from_low_{window}" for window in POSITION_WINDOWS],
    *[f"range_position_{window}" for window in POSITION_WINDOWS],
    "breakout_high_5", "breakout_high_20", "breakdown_low_5", "breakdown_low_20",
    "is_premarket", "is_regular", "is_afterhours",
    "minutes_from_regular_open", "minutes_to_regular_close",
    "is_opening_5m", "is_opening_30m", "is_power_hour",
    "minute_of_day_sin", "minute_of_day_cos", "day_of_week_sin", "day_of_week_cos",
]

assert len(MODEL_FEATURES) == len(set(MODEL_FEATURES)), "중복 feature 이름이 있습니다."
print(f"model feature count: {len(MODEL_FEATURES)}")
display(pd.DataFrame({"feature": MODEL_FEATURES}).head(20))


model feature count: 92


,feature
0,log_close_price
1,open_rel_prev_close
2,high_rel_prev_close
3,low_rel_prev_close
4,close_rel_prev_close
5,log_return_1
6,log_return_2
7,log_return_3
8,log_return_5
9,log_return_10


## 3. 가격 특징 계산 함수

함수는 하나의 연속 구간만 입력받는다. rolling과 EMA가 gap을 넘어 계산되지 않도록 파일을 먼저 연속 run으로 분리한다.

In [4]:
REQUIRED_COLUMNS = ["symbol", "timestamp_kst", "timestamp_utc", "open", "high", "low", "close"]

def safe_ratio(numerator: pd.Series, denominator: pd.Series) -> pd.Series:
    return numerator / denominator.where(denominator.abs() > 1e-12)

def compute_rsi(close: pd.Series, window: int) -> pd.Series:
    delta = close.diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)
    avg_gain = gain.ewm(alpha=1 / window, adjust=False, min_periods=window).mean()
    avg_loss = loss.ewm(alpha=1 / window, adjust=False, min_periods=window).mean()
    relative_strength = safe_ratio(avg_gain, avg_loss)
    rsi = 100 - 100 / (1 + relative_strength)
    rsi = rsi.mask((avg_gain > 0) & (avg_loss == 0), 100.0)
    rsi = rsi.mask((avg_gain == 0) & (avg_loss == 0), 50.0)
    return rsi

def load_price_file(path: Path) -> pd.DataFrame:
    header = pd.read_csv(path, nrows=0, encoding="utf-8-sig").columns
    missing = sorted(set(REQUIRED_COLUMNS) - set(header))
    if missing:
        raise ValueError(f"{path.name}: missing columns {missing}")

    frame = pd.read_csv(path, usecols=REQUIRED_COLUMNS, encoding="utf-8-sig")
    frame["symbol"] = frame["symbol"].astype(str).str.upper()
    frame["timestamp_utc"] = pd.to_datetime(frame["timestamp_utc"], errors="coerce", utc=True)
    frame["timestamp_kst"] = pd.to_datetime(frame["timestamp_kst"], errors="coerce")
    for column in ["open", "high", "low", "close"]:
        frame[column] = pd.to_numeric(frame[column], errors="coerce")
    frame = frame.sort_values(["symbol", "timestamp_utc"]).reset_index(drop=True)
    frame["source_row"] = np.arange(len(frame), dtype=np.int64)

    delta_minutes = frame.groupby("symbol")["timestamp_utc"].diff().dt.total_seconds().div(60)
    is_consecutive = pd.Series(np.isclose(delta_minutes, 1.0, rtol=0.0, atol=1e-6), index=frame.index)
    is_new_run = (~is_consecutive) | frame["symbol"].ne(frame["symbol"].shift())
    frame["run_number"] = is_new_run.cumsum().astype(np.int64)
    frame["run_id"] = path.stem + "::" + frame["run_number"].astype(str)
    return frame


In [5]:
def compute_price_features(run: pd.DataFrame) -> pd.DataFrame:
    out = run.copy()
    open_ = out["open"]
    high = out["high"]
    low = out["low"]
    close = out["close"]
    prev_close = close.shift(1)

    out["log_close_price"] = np.log(close.where(close > 0))
    out["open_rel_prev_close"] = safe_ratio(open_, prev_close) - 1
    out["high_rel_prev_close"] = safe_ratio(high, prev_close) - 1
    out["low_rel_prev_close"] = safe_ratio(low, prev_close) - 1
    out["close_rel_prev_close"] = safe_ratio(close, prev_close) - 1
    for lag in RETURN_LAGS:
        out[f"log_return_{lag}"] = np.log(safe_ratio(close, close.shift(lag)))

    candle_range = high - low
    body_abs = (close - open_).abs()
    upper_wick_abs = high - pd.concat([open_, close], axis=1).max(axis=1)
    lower_wick_abs = pd.concat([open_, close], axis=1).min(axis=1) - low
    out["body_return"] = safe_ratio(close - open_, open_)
    out["range_return"] = safe_ratio(candle_range, open_)
    out["upper_wick"] = safe_ratio(upper_wick_abs, open_)
    out["lower_wick"] = safe_ratio(lower_wick_abs, open_)
    out["close_location"] = safe_ratio(close - low, candle_range).where(candle_range > 0, 0.5)
    out["gap_from_prev_close"] = safe_ratio(open_, prev_close) - 1
    out["body_to_range_ratio"] = safe_ratio(body_abs, candle_range).where(candle_range > 0, 0.0)
    out["upper_wick_to_range"] = safe_ratio(upper_wick_abs, candle_range).where(candle_range > 0, 0.0)
    out["lower_wick_to_range"] = safe_ratio(lower_wick_abs, candle_range).where(candle_range > 0, 0.0)
    out["is_bullish"] = (close > open_).astype(float)
    out["is_bearish"] = (close < open_).astype(float)

    sma = {}
    for window in SMA_WINDOWS:
        sma[window] = close.rolling(window, min_periods=window).mean()
        out[f"close_to_sma{window}"] = safe_ratio(close, sma[window]) - 1
    ema = {}
    for window in EMA_WINDOWS:
        ema[window] = close.ewm(span=window, adjust=False, min_periods=window).mean()
        out[f"close_to_ema{window}"] = safe_ratio(close, ema[window]) - 1

    out["sma5_to_sma20"] = safe_ratio(sma[5], sma[20]) - 1
    out["sma10_to_sma30"] = safe_ratio(sma[10], sma[30]) - 1
    out["ema9_to_ema20"] = safe_ratio(ema[9], ema[20]) - 1
    out["ema12_to_ema26"] = safe_ratio(ema[12], ema[26]) - 1
    out["sma5_slope_3"] = safe_ratio(sma[5], sma[5].shift(3)) - 1
    out["sma20_slope_5"] = safe_ratio(sma[20], sma[20].shift(5)) - 1
    out["ema9_slope_3"] = safe_ratio(ema[9], ema[9].shift(3)) - 1
    out["ema20_slope_5"] = safe_ratio(ema[20], ema[20].shift(5)) - 1

    return_1 = np.log(safe_ratio(close, prev_close))
    for window in VOL_WINDOWS:
        out[f"rolling_std_return_{window}"] = return_1.rolling(window, min_periods=window).std(ddof=0)
    true_range = pd.concat([high - low, (high - prev_close).abs(), (low - prev_close).abs()], axis=1).max(axis=1)
    for window in ATR_WINDOWS:
        atr = true_range.rolling(window, min_periods=window).mean()
        out[f"atr_pct_{window}"] = safe_ratio(atr, close)

    bb_mid = close.rolling(20, min_periods=20).mean()
    bb_std = close.rolling(20, min_periods=20).std(ddof=0)
    bb_upper = bb_mid + 2 * bb_std
    bb_lower = bb_mid - 2 * bb_std
    bb_band = bb_upper - bb_lower
    out["bb_width_20"] = safe_ratio(bb_band, bb_mid)
    out["bb_position_20"] = safe_ratio(close - bb_lower, bb_band).where(bb_band > 0, 0.5)

    macd = ema[12] - ema[26]
    macd_signal = macd.ewm(span=9, adjust=False, min_periods=9).mean()
    out["macd_scaled"] = safe_ratio(macd, close)
    out["macd_signal_scaled"] = safe_ratio(macd_signal, close)
    out["macd_hist_scaled"] = safe_ratio(macd - macd_signal, close)

    for lag in [3, 5, 10, 20]:
        out[f"roc_{lag}"] = safe_ratio(close, close.shift(lag)) - 1
    out["rsi_6"] = compute_rsi(close, 6)
    out["rsi_14"] = compute_rsi(close, 14)
    stochastic_low = low.rolling(14, min_periods=14).min()
    stochastic_high = high.rolling(14, min_periods=14).max()
    stochastic_range = stochastic_high - stochastic_low
    stochastic_k = safe_ratio(close - stochastic_low, stochastic_range).where(stochastic_range > 0, 0.5)
    out["stochastic_k_14"] = stochastic_k
    out["stochastic_d_3"] = stochastic_k.rolling(3, min_periods=3).mean()

    for window in POSITION_WINDOWS:
        rolling_high = high.rolling(window, min_periods=window).max()
        rolling_low = low.rolling(window, min_periods=window).min()
        rolling_range = rolling_high - rolling_low
        out[f"distance_from_high_{window}"] = safe_ratio(close, rolling_high) - 1
        out[f"distance_from_low_{window}"] = safe_ratio(close, rolling_low) - 1
        out[f"range_position_{window}"] = safe_ratio(close - rolling_low, rolling_range).where(rolling_range > 0, 0.5)
    for window in [5, 20]:
        prior_high = high.shift(1).rolling(window, min_periods=window).max()
        prior_low = low.shift(1).rolling(window, min_periods=window).min()
        out[f"breakout_high_{window}"] = (high > prior_high).astype(float).where(prior_high.notna())
        out[f"breakdown_low_{window}"] = (low < prior_low).astype(float).where(prior_low.notna())

    timestamp_et = out["timestamp_utc"].dt.tz_convert("America/New_York")
    minute_of_day = timestamp_et.dt.hour * 60 + timestamp_et.dt.minute
    day_of_week = timestamp_et.dt.dayofweek
    out["is_premarket"] = ((minute_of_day >= 240) & (minute_of_day < 570)).astype(float)
    out["is_regular"] = ((minute_of_day >= 570) & (minute_of_day < 960)).astype(float)
    out["is_afterhours"] = ((minute_of_day >= 960) & (minute_of_day < 1200)).astype(float)
    out["minutes_from_regular_open"] = minute_of_day - 570
    out["minutes_to_regular_close"] = 960 - minute_of_day
    out["is_opening_5m"] = ((minute_of_day >= 570) & (minute_of_day < 575)).astype(float)
    out["is_opening_30m"] = ((minute_of_day >= 570) & (minute_of_day < 600)).astype(float)
    out["is_power_hour"] = ((minute_of_day >= 900) & (minute_of_day < 960)).astype(float)
    out["minute_of_day_sin"] = np.sin(2 * np.pi * minute_of_day / 1440)
    out["minute_of_day_cos"] = np.cos(2 * np.pi * minute_of_day / 1440)
    out["day_of_week_sin"] = np.sin(2 * np.pi * day_of_week / 5)
    out["day_of_week_cos"] = np.cos(2 * np.pi * day_of_week / 5)

    out[MODEL_FEATURES] = out[MODEL_FEATURES].replace([np.inf, -np.inf], np.nan)
    return out


## 4. 전체 파일 feature 생성과 60봉 sequence index

`feature_frames`는 이번 노트북 검증을 위한 메모리 객체다. 실제 학습 파이프라인에서는 동일 함수를 모듈화하고 lazy dataset으로 읽는다.

In [6]:
def build_file_features(path: Path) -> tuple[pd.DataFrame, dict]:
    raw = load_price_file(path)
    run_lengths = raw.groupby("run_id").size()
    eligible_run_ids = set(run_lengths[run_lengths >= MIN_RUN_LENGTH_FOR_SEQUENCE].index)
    parts = []
    for run_id, run in raw.groupby("run_id", sort=False):
        if run_id in eligible_run_ids:
            parts.append(compute_price_features(run.reset_index(drop=True)))
    if parts:
        result = pd.concat(parts, ignore_index=True).copy()  # feature 열 단편화 해소
    else:
        result = pd.DataFrame(columns=[*raw.columns, *MODEL_FEATURES])
    result["feature_row"] = np.arange(len(result), dtype=np.int64)
    result["source_path"] = str(path)
    result["session"] = path.parent.name
    result["split"] = SPLIT_BY_SESSION[path.parent.name]
    source_stats = {
        "symbol": raw["symbol"].iloc[0] if len(raw) else None,
        "raw_rows": len(raw),
        "continuous_runs": len(run_lengths),
        "eligible_runs": len(eligible_run_ids),
        "longest_run": int(run_lengths.max()) if len(run_lengths) else 0,
    }
    return result, source_stats

def valid_blocks(mask: pd.Series):
    group_id = mask.ne(mask.shift(fill_value=False)).cumsum()
    for _, index in mask[mask].groupby(group_id[mask]).groups.items():
        positions = np.asarray(list(index), dtype=np.int64)
        if len(positions):
            yield int(positions[0]), int(positions[-1])

feature_frames: dict[str, pd.DataFrame] = {}
file_audit_rows: list[dict] = []
sequence_rows: list[dict] = []

for path in ENRICHED_PATHS:
    features, source_stats = build_file_features(path)
    path_key = str(path)
    feature_frames[path_key] = features
    feature_valid = features[MODEL_FEATURES].notna().all(axis=1)
    sequence_count = 0

    for run_id, run in features.groupby("run_id", sort=False):
        run_valid = feature_valid.loc[run.index]
        for start_row, end_row in valid_blocks(run_valid):
            block_length = end_row - start_row + 1
            if block_length < SEQUENCE_LENGTH:
                continue
            for sequence_end in range(start_row + SEQUENCE_LENGTH - 1, end_row + 1):
                sequence_start = sequence_end - SEQUENCE_LENGTH + 1
                end_record = features.loc[sequence_end]
                start_record = features.loc[sequence_start]
                sequence_rows.append({
                    "source_path": path_key,
                    "session": path.parent.name,
                    "split": SPLIT_BY_SESSION[path.parent.name],
                    "symbol": end_record["symbol"],
                    "run_id": run_id,
                    "start_feature_row": sequence_start,
                    "end_feature_row": sequence_end,
                    "input_start_timestamp": start_record["timestamp_utc"],
                    "input_end_timestamp": end_record["timestamp_utc"],
                    "feature_version": "price_v1",
                })
                sequence_count += 1

    file_audit_rows.append({
        "source_path": path_key,
        "session": path.parent.name,
        "split": SPLIT_BY_SESSION[path.parent.name],
        "symbol": source_stats["symbol"],
        "raw_rows": source_stats["raw_rows"],
        "continuous_runs": source_stats["continuous_runs"],
        "eligible_runs": source_stats["eligible_runs"],
        "longest_run": source_stats["longest_run"],
        "fully_valid_feature_rows": int(feature_valid.sum()),
        "sequence_count": sequence_count,
    })

file_audit_df = pd.DataFrame(file_audit_rows)
sequence_index_df = pd.DataFrame(sequence_rows)

print(f"feature files: {len(feature_frames):,}")
print(f"sequence index rows: {len(sequence_index_df):,}")
display(file_audit_df.head())
display(sequence_index_df.head())


feature files: 170
sequence index rows: 42,162


,source_path,session,split,symbol,raw_rows,continuous_runs,eligible_runs,longest_run,fully_valid_feature_rows,sequence_count
0,/home/user/urbandatalab/YSLee/data/stock_data/...,session_2026-07-07,train,ALM,446,38,1,396,336,277
1,/home/user/urbandatalab/YSLee/data/stock_data/...,session_2026-07-07,train,BMNR,736,82,1,443,383,324
2,/home/user/urbandatalab/YSLee/data/stock_data/...,session_2026-07-07,train,BURU,759,84,2,146,156,38
3,/home/user/urbandatalab/YSLee/data/stock_data/...,session_2026-07-07,train,CLRO,695,62,2,378,411,293
4,/home/user/urbandatalab/YSLee/data/stock_data/...,session_2026-07-07,train,DCX,165,37,1,123,63,4


,source_path,session,split,symbol,run_id,start_feature_row,end_feature_row,input_start_timestamp,input_end_timestamp,feature_version
0,/home/user/urbandatalab/YSLee/data/stock_data/...,session_2026-07-07,train,ALM,alm_2026-07-07_1700-0800_kst_7ff6d430_enriched...,60,119,2026-07-07 14:25:00+00:00,2026-07-07 15:24:00+00:00,price_v1
1,/home/user/urbandatalab/YSLee/data/stock_data/...,session_2026-07-07,train,ALM,alm_2026-07-07_1700-0800_kst_7ff6d430_enriched...,61,120,2026-07-07 14:26:00+00:00,2026-07-07 15:25:00+00:00,price_v1
2,/home/user/urbandatalab/YSLee/data/stock_data/...,session_2026-07-07,train,ALM,alm_2026-07-07_1700-0800_kst_7ff6d430_enriched...,62,121,2026-07-07 14:27:00+00:00,2026-07-07 15:26:00+00:00,price_v1
3,/home/user/urbandatalab/YSLee/data/stock_data/...,session_2026-07-07,train,ALM,alm_2026-07-07_1700-0800_kst_7ff6d430_enriched...,63,122,2026-07-07 14:28:00+00:00,2026-07-07 15:27:00+00:00,price_v1
4,/home/user/urbandatalab/YSLee/data/stock_data/...,session_2026-07-07,train,ALM,alm_2026-07-07_1700-0800_kst_7ff6d430_enriched...,64,123,2026-07-07 14:29:00+00:00,2026-07-07 15:28:00+00:00,price_v1


## 5. Feature 결측과 sequence 생성 결과

In [7]:
total_rows = sum(len(frame) for frame in feature_frames.values())
nan_counts = sum((frame[MODEL_FEATURES].isna().sum() for frame in feature_frames.values()), start=pd.Series(0, index=MODEL_FEATURES, dtype=np.int64))
nan_summary = pd.DataFrame({
    "nan_count": nan_counts,
    "nan_share": nan_counts / max(total_rows, 1),
}).sort_values(["nan_share", "nan_count"], ascending=False)

sequence_summary = pd.Series({
    "feature_computed_rows": total_rows,
    "feature_count": len(MODEL_FEATURES),
    "files_with_sequences": int(file_audit_df["sequence_count"].gt(0).sum()),
    "files_without_sequences": int(file_audit_df["sequence_count"].eq(0).sum()),
    "total_sequences": len(sequence_index_df),
}, name="value")
display(sequence_summary.to_frame())
display(nan_summary.head(20))

session_sequences = sequence_index_df.groupby(["split", "session"], observed=True).size().rename("sequences").reset_index()
display(session_sequences)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
session_sequences.plot.bar(x="session", y="sequences", ax=axes[0], legend=False, title="세션별 60봉 sequence 수")
file_audit_df["longest_run"].plot.hist(bins=30, ax=axes[1], title="파일별 최장 연속 1분봉 길이")
plt.tight_layout()
plt.show()


,value
feature_computed_rows,62511
feature_count,92
files_with_sequences,128
files_without_sequences,42
total_sequences,42162


,nan_count,nan_share
rolling_std_return_60,10260,0.164131
close_to_sma60,10089,0.161396
distance_from_high_60,10089,0.161396
distance_from_low_60,10089,0.161396
macd_signal_scaled,5643,0.090272
macd_hist_scaled,5643,0.090272
close_to_sma30,4959,0.079330
sma10_to_sma30,4959,0.079330
close_to_ema26,4275,0.068388
ema12_to_ema26,4275,0.068388


,split,session,sequences
0,test,session_2026-07-15,6895
1,train,session_2026-07-07,2823
2,train,session_2026-07-08,7295
3,train,session_2026-07-09,7008
4,train,session_2026-07-10,7166
5,validation,session_2026-07-13,6129
6,validation,session_2026-07-14,4846


## 6. 필수 검증: 금지 feature와 미래정보 누수

In [8]:
FORBIDDEN_PATTERNS = [
    "volume", "notional", "trade_count", "vwap", "quote", "bid", "ask",
    "spread", "imbalance", "aggressive", "sell_pressure", "trade_strength", "tick_count",
]
forbidden_features = [
    feature for feature in MODEL_FEATURES
    if any(pattern in feature.lower() for pattern in FORBIDDEN_PATTERNS)
]
assert not forbidden_features, f"금지 feature 발견: {forbidden_features}"
assert set(MODEL_FEATURES) <= set(next(iter(feature_frames.values())).columns)
print("PASS: provider-dependent 금지 feature가 없습니다.")


PASS: provider-dependent 금지 feature가 없습니다.


In [9]:
leakage_candidate = file_audit_df.sort_values("longest_run", ascending=False).iloc[0]
leakage_path = Path(leakage_candidate["source_path"])
raw_for_test = load_price_file(leakage_path)
run_sizes = raw_for_test.groupby("run_id").size().sort_values(ascending=False)
test_run_id = run_sizes.index[0]
test_run = raw_for_test[raw_for_test["run_id"] == test_run_id].reset_index(drop=True)
decision_position = max(60, min(len(test_run) - 2, len(test_run) // 2))

base_features = compute_price_features(test_run)
mutated_run = test_run.copy()
future_mask = mutated_run.index > decision_position
mutated_run.loc[future_mask, ["open", "high", "low", "close"]] *= 7.0
mutated_features = compute_price_features(mutated_run)

base_past = base_features.loc[:decision_position, MODEL_FEATURES].to_numpy(dtype=float)
mutated_past = mutated_features.loc[:decision_position, MODEL_FEATURES].to_numpy(dtype=float)
np.testing.assert_allclose(base_past, mutated_past, rtol=0.0, atol=0.0, equal_nan=True)
print(f"PASS: t+1 이후 OHLC를 변경해도 t까지의 feature가 변하지 않습니다. ({leakage_path.name})")


PASS: t+1 이후 OHLC를 변경해도 t까지의 feature가 변하지 않습니다. (skyq_2026-07-07_1700-0800_kst_3ad80c83_enriched.csv)


## 7. Sequence materialization 표본 검증

In [10]:
def materialize_sequence(metadata: pd.Series) -> np.ndarray:
    frame = feature_frames[metadata["source_path"]]
    start = int(metadata["start_feature_row"])
    end = int(metadata["end_feature_row"])
    sequence = frame.loc[start:end, MODEL_FEATURES].to_numpy(dtype=np.float32)
    timestamps = frame.loc[start:end, "timestamp_utc"]
    symbols = frame.loc[start:end, "symbol"]
    delta_minutes = timestamps.diff().dt.total_seconds().div(60).dropna()

    assert sequence.shape == (SEQUENCE_LENGTH, len(MODEL_FEATURES))
    assert np.isfinite(sequence).all()
    assert symbols.nunique() == 1
    assert np.isclose(delta_minutes.to_numpy(), 1.0, rtol=0.0, atol=1e-6).all()
    assert timestamps.iloc[0] == metadata["input_start_timestamp"]
    assert timestamps.iloc[-1] == metadata["input_end_timestamp"]
    return sequence

sample_size = min(128, len(sequence_index_df))
sample_metadata = sequence_index_df.sample(sample_size, random_state=42).reset_index(drop=True)
sample_batch = np.stack([materialize_sequence(row) for _, row in sample_metadata.iterrows()])

assert sample_batch.shape == (sample_size, SEQUENCE_LENGTH, len(MODEL_FEATURES))
print(f"PASS: sample batch shape = {sample_batch.shape}, dtype = {sample_batch.dtype}")


PASS: sample batch shape = (128, 60, 92), dtype = float32


## 8. 날짜 split과 train-only scaler

현재 7개 세션 split은 파이프라인 검증용이다. 실제 모델 성능을 확정하기에는 날짜 수가 부족하다. scaler는 겹치는 sequence를 반복 가중하지 않도록 **train의 고유 bar**에만 fit한다.

In [11]:
split_summary = sequence_index_df.groupby("split").agg(
    sequences=("symbol", "size"),
    symbols=("symbol", "nunique"),
    sessions=("session", "nunique"),
    start=("input_start_timestamp", "min"),
    end=("input_end_timestamp", "max"),
).reindex(["train", "validation", "test"])
display(split_summary)

train_feature_rows = []
for frame in feature_frames.values():
    if frame.empty or frame["split"].iloc[0] != "train":
        continue
    valid = frame[MODEL_FEATURES].notna().all(axis=1)
    train_feature_rows.append(frame.loc[valid, MODEL_FEATURES])
train_feature_df = pd.concat(train_feature_rows, ignore_index=True)

scaler = RobustScaler(quantile_range=(25.0, 75.0))
scaler.fit(train_feature_df[MODEL_FEATURES].to_numpy(dtype=np.float64))
scaled_sample_batch = scaler.transform(sample_batch.reshape(-1, len(MODEL_FEATURES))).reshape(sample_batch.shape).astype(np.float32)

assert np.isfinite(scaler.center_).all()
assert np.isfinite(scaler.scale_).all()
assert np.isfinite(scaled_sample_batch).all()
print(f"PASS: scaler fit rows = {len(train_feature_df):,} unique train bars")
print(f"scaled sample batch shape = {scaled_sample_batch.shape}")

scaler_audit = pd.DataFrame({
    "feature": MODEL_FEATURES,
    "train_median": scaler.center_,
    "train_iqr": scaler.scale_,
}).sort_values("train_iqr")
display(scaler_audit.head(15))


,sequences,symbols,sessions,start,end
split,,,,,
train,24292,54,4,2026-07-07 09:00:00+00:00,2026-07-10 23:00:00+00:00
validation,10975,27,2,2026-07-13 09:00:00+00:00,2026-07-14 23:00:00+00:00
test,6895,20,1,2026-07-15 09:00:00+00:00,2026-07-15 23:00:00+00:00


PASS: scaler fit rows = 30,605 unique train bars
scaled sample batch shape = (128, 60, 92)


,feature,train_median,train_iqr
1,open_rel_prev_close,0.000000e+00,0.001592
16,gap_from_prev_close,0.000000e+00,0.001592
55,macd_hist_scaled,4.441125e-05,0.002921
14,lower_wick,8.561644e-04,0.004348
13,upper_wick,8.477450e-04,0.004405
28,close_to_ema3,-1.522310e-04,0.004599
22,close_to_sma3,-1.110223e-16,0.006027
40,ema9_slope_3,-4.202448e-04,0.006938
29,close_to_ema5,-3.149255e-04,0.007028
36,ema9_to_ema20,-7.544766e-04,0.007853


## 9. Phase 2 결론

In [12]:
PROCESSED_ROOT.mkdir(parents=True, exist_ok=True)
FEATURE_ROWS_PATH = PROCESSED_ROOT / "feature_rows_price_v1.parquet"
SEQUENCE_INDEX_PATH = PROCESSED_ROOT / "sequence_index_price_v1.parquet"
FEATURE_AUDIT_PATH = PROCESSED_ROOT / "feature_file_audit_price_v1.parquet"
SCALER_AUDIT_PATH = PROCESSED_ROOT / "bar_scaler_audit_price_v1.parquet"

feature_row_metadata = ["source_path", "session", "split", "symbol", "run_id", "feature_row", "timestamp_utc"]
feature_rows_df = pd.concat(
    [frame[feature_row_metadata + MODEL_FEATURES] for frame in feature_frames.values() if not frame.empty],
    ignore_index=True,
)
feature_rows_df[MODEL_FEATURES] = feature_rows_df[MODEL_FEATURES].astype(np.float32)
feature_rows_df.to_parquet(FEATURE_ROWS_PATH, index=False, compression="zstd")
sequence_index_df.to_parquet(SEQUENCE_INDEX_PATH, index=False, compression="zstd")
file_audit_df.to_parquet(FEATURE_AUDIT_PATH, index=False, compression="zstd")
scaler_audit.to_parquet(SCALER_AUDIT_PATH, index=False, compression="zstd")
print(f"saved: {FEATURE_ROWS_PATH} ({len(feature_rows_df):,} rows)")
print(f"saved: {SEQUENCE_INDEX_PATH} ({len(sequence_index_df):,} rows)")

split_counts = sequence_index_df["split"].value_counts()
findings = [
    f"**입력 schema:** 공급자 독립적인 가격·시간 feature {len(MODEL_FEATURES)}개를 확정했다.",
    f"**60봉 표본:** 기술지표 warm-up과 gap 제외 후 {len(sequence_index_df):,}개 sequence를 생성할 수 있다.",
    f"**파일 coverage:** {int(file_audit_df['sequence_count'].gt(0).sum())}/{len(file_audit_df)}개 파일에서 학습 가능한 sequence가 생성된다.",
    f"**임시 split:** train {int(split_counts.get('train', 0)):,}, validation {int(split_counts.get('validation', 0)):,}, test {int(split_counts.get('test', 0)):,}개다.",
    "**검증 통과:** 금지 feature, 미래정보 누수, sequence 길이·종목·시간 연속성, train-only scaler 검사를 통과했다.",
    f"**저장 완료:** bar feature와 sequence index를 `{PROCESSED_ROOT}`에 parquet으로 저장했다.",
    "**다음 단계:** 저장된 sequence index와 10분 TP/SL 라벨을 결합해 Torch baseline을 학습한다.",
]
display(Markdown("### 실행 결론\n\n" + "\n\n".join(f"- {item}" for item in findings)))


saved: /home/user/urbandatalab/YSLee/data/stock_data/processed/feature_rows_price_v1.parquet (62,511 rows)
saved: /home/user/urbandatalab/YSLee/data/stock_data/processed/sequence_index_price_v1.parquet (42,162 rows)


### 실행 결론

- **입력 schema:** 공급자 독립적인 가격·시간 feature 92개를 확정했다.

- **60봉 표본:** 기술지표 warm-up과 gap 제외 후 42,162개 sequence를 생성할 수 있다.

- **파일 coverage:** 128/170개 파일에서 학습 가능한 sequence가 생성된다.

- **임시 split:** train 24,292, validation 10,975, test 6,895개다.

- **검증 통과:** 금지 feature, 미래정보 누수, sequence 길이·종목·시간 연속성, train-only scaler 검사를 통과했다.

- **저장 완료:** bar feature와 sequence index를 `/home/user/urbandatalab/YSLee/data/stock_data/processed`에 parquet으로 저장했다.

- **다음 단계:** 저장된 sequence index와 10분 TP/SL 라벨을 결합해 Torch baseline을 학습한다.

### 후속 구현 체크리스트

- [ ] 검증된 feature 함수를 `src/features/price_features.py`로 이동
- [ ] sequence index builder를 재사용 가능한 모듈로 이동
- [x] feature row, sequence index, audit artifact를 parquet으로 저장
- [x] 미래 10분이 충분한 sequence만 라벨 단계에서 선별
- [x] 수수료·진입가·TP/SL barrier 단위 테스트 작성
- [x] Dual-path 확정/ambiguous 라벨 생성
